Cell 1 — Import Library & Set Seed Mengimpor semua library yang dibutuhkan: PyTorch untuk deep learning, Transformers (HuggingFace) untuk IndoBERT, dan scikit-learn untuk split data serta metrik evaluasi. `SEED = 42` diterapkan ke semua random generator (Python, NumPy, PyTorch) supaya hasil training bisa direproduksi persis sama kalau notebook ini dijalankan ulang. `DEVICE` otomatis mendeteksi dan memilih GPU (CUDA) kalau tersedia, karena training di CPU akan jauh lebih lambat.

In [ ]:
import os
import re
import random
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    AutoTokenizer,
    AutoModel,
    get_linear_schedule_with_warmup,
    set_seed,
)

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
set_seed(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print("Device:", DEVICE)
print("PyTorch version:", torch.__version__)

Device: cuda
PyTorch version: 2.11.0+cu128


Cell 2 — Install Library Transformers Colab tidak menyediakan library `transformers` secara default, jadi perlu diinstall manual. Flag `-q` (quiet) supaya log instalasi tidak memenuhi output.

In [ ]:
!pip install transformers -q

Cell 3 — Load Dataset Hasil RelabelingMembaca file CSV yang labelnya sudah dihasilkan dari model pretrained `mdhugol/indonesia-bert-sentiment-classification` (bukan dari rating bintang). Baris berlabel "netral" dibuang karena penelitian ini fokus ke klasifikasi biner sesuai Bab 3.3 skripsi. Dicetak juga distribusi label rating-based (`label_original`, sekadar pembanding/referensi, tidak dipakai untuk training) dan distribusi jumlah data per kategori aplikasi, untuk memastikan ke-17 kategori benar-benar terwakili di dataset.

In [ ]:
DATA_PATH = "ulasan_aplikasi_50000_relabeled.csv"
TEXT_COL  = "ulasan"
LABEL_COL = "label"
KATEGORI_COL = "kategori"

df = pd.read_csv(DATA_PATH)
print("=" * 60)
print("DATASET: Multi-Kategori Relabeled (v5 - Binary)")
print("=" * 60)
print(f"Original Shape: {df.shape}")

# Filter out neutral reviews for binary classification (konsisten dengan v4)
df = df[df[LABEL_COL].isin(["positif", "negatif", "positive", "negative"])].copy()
print(f"Shape after filtering 'netral': {df.shape}")

print(f"\nDistribusi label (text-based):")
print(df[LABEL_COL].value_counts())
print(f"\nLabel asli (rating-based) -- hanya referensi:")
print(df["label_original"].value_counts())
print(f"\nDistribusi per kategori:")
print(df[KATEGORI_COL].value_counts())
print(f"\nSample data:")
df[[TEXT_COL, "skor", LABEL_COL, "label_original", KATEGORI_COL, "sentiment_score"]].head(5)

DATASET: Multi-Kategori Relabeled (v5 - Binary)
Original Shape: (46978, 12)
Shape after filtering 'netral': (42138, 12)

Distribusi label (text-based):
label
positif    24762
negatif    17376
Name: count, dtype: int64

Label asli (rating-based) -- hanya referensi:
label_original
positif    26712
negatif    13582
netral      1844
Name: count, dtype: int64

Distribusi per kategori:
kategori
game_casual_puzzle       2793
transportasi_travel      2705
ecommerce                2701
food_delivery            2699
kesehatan_medis          2687
game_action_moba         2687
perbankan                2640
sosial                   2636
keuangan_investasi       2632
hiburan_streaming        2614
berita                   2567
game_battle_royale       2564
dompet_digital           2563
komunikasi               2560
gaya_hidup_kecantikan    2558
pendidikan               1385
produktivitas            1147
Name: count, dtype: int64

Sample data:


,ulasan,skor,label,label_original,kategori,sentiment_score
1,bagus,5,positif,positif,ecommerce,0.996109
2,aku kecewa banget dri kmrn ada 4x co selalu di...,1,negatif,negatif,ecommerce,0.997659
3,mantap,5,positif,positif,ecommerce,0.997200
4,baik,1,positif,negatif,ecommerce,0.984418
5,ok,5,positif,positif,ecommerce,0.944405


 Cell 4 — Preprocessing Teks (Emoji Mapping, Slang Normalization, Dedup) Mendefinisikan kamus normalisasi slang (60+ kata gaul → kata baku, mis. "gk" → "tidak") dan pemetaan emoji ke teks bermakna sentimen (mis. 👍 → "bagus"). Fungsi `clean_text()` menjalankan seluruh pipeline: emoji mapping, lowercase, hapus URL/mention/HTML, normalisasi huruf berulang ("bagusss" → "baguss"), lalu normalisasi slang. Setelah dibersihkan, baris dengan teks yang jadi identik di-*drop_duplicates* — langkah ini penting supaya tidak ada ulasan yang persis sama muncul di lebih dari satu tempat, mencegah potensi kebocoran data antara train dan test nanti.

In [ ]:
SLANG_MAP = {
    "gk": "tidak", "ga": "tidak", "gak": "tidak", "nggak": "tidak",
    "ngga": "tidak", "tdk": "tidak", "engga": "tidak", "enggak": "tidak",
    "kagak": "tidak", "kaga": "tidak", "ndak": "tidak", "nda": "tidak",
    "bgt": "banget", "bgtt": "banget", "bngt": "banget",
    "skl": "sekali",
    "apk": "aplikasi", "app": "aplikasi", "apps": "aplikasi",
    "dr": "dari", "drpd": "daripada", "dgn": "dengan", "dg": "dengan",
    "sm": "sama", "brsm": "bersama", "pd": "pada", "utk": "untuk",
    "tuk": "untuk", "buat": "untuk",
    "krn": "karena", "karna": "karena", "krna": "karena",
    "yg": "yang", "tp": "tapi", "tpi": "tapi", "ttp": "tetap",
    "ttpi": "tetapi", "ttg": "tentang",
    "sy": "saya", "gw": "saya", "gue": "saya",
    "km": "kamu", "lo": "kamu", "lu": "kamu",
    "udah": "sudah", "udh": "sudah", "dah": "sudah", "sdh": "sudah",
    "blm": "belum", "blum": "belum",
    "skrg": "sekarang", "skrng": "sekarang",
    "lg": "lagi", "lgi": "lagi",
    "trs": "terus", "trus": "terus",
    "msh": "masih", "masi": "masih",
    "hbs": "habis",
    "aja": "saja", "aj": "saja",
    "emg": "memang", "emang": "memang",
    "nih": "ini", "tuh": "itu",
    "bkn": "bukan",
    "klo": "kalau", "klu": "kalau", "kl": "kalau", "klau": "kalau",
    "gimana": "bagaimana", "gmn": "bagaimana",
    "knp": "kenapa",
    "lbh": "lebih",
    "kyk": "seperti", "kyak": "seperti", "kayak": "seperti",
    "jd": "jadi", "jdi": "jadi",
    "bs": "bisa", "bsa": "bisa",
    "hrs": "harus",
    "jg": "juga",
    "mau": "mau", "mo": "mau",
    "dpt": "dapat", "dpat": "dapat",
    "sdkt": "sedikit",
    "sampe": "sampai", "ampe": "sampai",
    "bentar": "sebentar",
    "pake": "pakai", "pk": "pakai",
    "nunggu": "menunggu",
    "nyari": "mencari",
    "tmn": "teman",
    "ok": "oke",
    "mantap": "bagus", "mantul": "bagus", "kece": "bagus",
    "jelek": "jelek", "buruk": "buruk", "parah": "parah",
    "rugi": "rugi", "kecewa": "kecewa", "puas": "puas",
}

EMOJI_MAP = {
    "\U0001F44D": " bagus ", "\U0001F44E": " jelek ", "\U0001F621": " kecewa ", "\U0001F62D": " sedih ",
    "\U0001F60A": " senang ", "\U0001F60D": " suka ", "\u2764\uFE0F": " cinta ", "\U0001F31F": " mantap ",
    "\U0001F622": " sedih ", "\U0001F620": " marah ", "\U0001F612": " kesal ", "\U0001F618": " suka ",
    "\U0001F601": " senang ", "\U0001F44C": " oke ", "\U0001F44F": " bagus ", "\U0001F496": " cinta ",
    "\U0001F389": " senang ", "\U0001F610": " biasa ", "\U0001F914": " biasa ", "\U0001F937": " biasa "
}

def replace_emojis(text: str) -> str:
    for emo, txt in EMOJI_MAP.items():
        text = text.replace(emo, txt)
    return text

def normalize_repeated_chars(text: str) -> str:
    return re.sub(r"(.)\1{2,}", r"\1\1", text)

def replace_slang(text: str) -> str:
    tokens = text.split()
    return " ".join([SLANG_MAP.get(tok, tok) for tok in tokens])

def clean_text(text: str) -> str:
    if not isinstance(text, str):
        text = "" if pd.isna(text) else str(text)
    text = replace_emojis(text)
    text = text.lower()
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"[^\w\s.,!?;:'\"()\-/%]", " ", text, flags=re.UNICODE)
    text = normalize_repeated_chars(text)
    text = replace_slang(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df[TEXT_COL] = df[TEXT_COL].astype(str).apply(clean_text)
df = df[df[TEXT_COL].str.len() > 0].copy()
df = df.drop_duplicates(subset=[TEXT_COL]).reset_index(drop=True)
print(f"After cleaning shape: {df.shape}")

After cleaning shape: (28981, 12)


Cell 5 — Konversi Label ke Angka Label teks ("positif"/"negatif") dikonversi jadi angka (0 = negative, 1 = positive) karena PyTorch memproses loss dan prediksi dalam bentuk numerik, bukan string.

In [ ]:
LABEL_MAP = {
    "negative": 0, "negatif": 0, 0: 0, "0": 0,
    "positive": 1, "positif": 1, 1: 1, "1": 1,
}
ID2LABEL  = {0: "negative", 1: "positive"}
LABEL2ID  = {"negative": 0, "positive": 1}

def map_label(x):
    if x in LABEL_MAP:
        return LABEL_MAP[x]
    x = str(x).strip().lower()
    if x in LABEL_MAP:
        return LABEL_MAP[x]
    raise ValueError(f"Label tidak dikenali: {x}")

df["label_id"] = df[LABEL_COL].apply(map_label)
print("Distribusi label_id:")
print(df["label_id"].value_counts().sort_index())

Distribusi label_id:
label_id
0    15735
1    13246
Name: count, dtype: int64


Cell 6 — Split Data 80/10/10 (Stratified) Dataset dibagi jadi tiga bagian: 80% data latih (train), 10% data validasi (dipakai memantau performa & early stopping selama training), dan 10% data uji (test, dievaluasi sekali di akhir, tidak pernah dilihat model sebelumnya). Pembagian dilakukan *stratified* berdasarkan `label_id`, supaya proporsi kelas positif/negatif tetap konsisten sama persis di ketiga subset — mencegah salah satu subset kebetulan didominasi satu kelas saja.

In [ ]:
train_temp, test_df = train_test_split(
    df, test_size=0.10, random_state=SEED, stratify=df["label_id"]
)
val_ratio = 0.10 / 0.90
train_df, val_df = train_test_split(
    train_temp, test_size=val_ratio, random_state=SEED, stratify=train_temp["label_id"]
)

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f"Train : {train_df.shape}  | dist: {train_df['label_id'].value_counts().sort_index().to_dict()}")
print(f"Val   : {val_df.shape}   | dist: {val_df['label_id'].value_counts().sort_index().to_dict()}")
print(f"Test  : {test_df.shape}   | dist: {test_df['label_id'].value_counts().sort_index().to_dict()}")

print("\n--- Distribusi kategori di data Test (laporan tambahan) ---")
print(test_df[KATEGORI_COL].value_counts())

Train : (23184, 13)  | dist: {0: 12588, 1: 10596}
Val   : (2898, 13)   | dist: {0: 1573, 1: 1325}
Test  : (2899, 13)   | dist: {0: 1574, 1: 1325}

--- Distribusi kategori di data Test (laporan tambahan) ---
kategori
game_action_moba         266
ecommerce                215
komunikasi               212
game_battle_royale       204
hiburan_streaming        201
keuangan_investasi       199
dompet_digital           183
gaya_hidup_kecantikan    180
sosial                   179
transportasi_travel      178
perbankan                177
kesehatan_medis          167
game_casual_puzzle       162
berita                   157
pendidikan               122
produktivitas             97
Name: count, dtype: int64


Cell 7 — Hitung Class Weights Karena jumlah data positif dan negatif tidak sama persis (imbalanced), dihitung bobot kelas otomatis (`class_weight='balanced'`, BUKAN SMOTE) — supaya saat training, kesalahan pada kelas yang lebih sedikit datanya "dihitung lebih berat" oleh loss function. Ini mencegah model cuma condong ke kelas mayoritas demi angka akurasi yang semu.

In [ ]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=train_df["label_id"].values,
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(DEVICE)
print(f"Class weights: {class_weights_tensor}")
print(f"  negative (0): {class_weights[0]:.4f}")
print(f"  positive (1): {class_weights[1]:.4f}")

Class weights: tensor([0.9209, 1.0940], device='cuda:0')
  negative (0): 0.9209
  positive (1): 1.0940


Cell 8 — Load Tokenizer IndoBERT Memuat tokenizer resmi dari model pretrained `indobenchmark/indobert-base-p1` — tokenizer inilah yang mengubah teks jadi deretan angka (token ID) sesuai vocabulary yang dipakai saat IndoBERT di-*pretrain*. `MAX_LENGTH = 256` artinya tiap ulasan dipotong (kalau lebih panjang) atau di-*pad* (kalau lebih pendek) supaya semua input punya panjang seragam 256 token.

In [ ]:
MODEL_NAME = "indobenchmark/indobert-base-p1"
MAX_LENGTH = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Tokenizer loaded: {MODEL_NAME}")

config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/229k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Tokenizer loaded: indobenchmark/indobert-base-p1


Cell 9 — Dataset Class & Persiapan DataLoader `SentimentDataset` adalah class pembungkus supaya data teks & label bisa dipakai `DataLoader` PyTorch — setiap kali dipanggil, otomatis melakukan tokenisasi satu baris teks dan mengembalikan `input_ids`, `attention_mask`, dan `label` dalam format tensor yang siap masuk ke model.

In [ ]:
class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts      = texts
        self.labels     = labels
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels":         torch.tensor(self.labels[idx], dtype=torch.long),
        }

train_dataset = SentimentDataset(train_df[TEXT_COL].tolist(), train_df["label_id"].tolist(), tokenizer, MAX_LENGTH)
val_dataset   = SentimentDataset(val_df[TEXT_COL].tolist(),   val_df["label_id"].tolist(),   tokenizer, MAX_LENGTH)
test_dataset  = SentimentDataset(test_df[TEXT_COL].tolist(),  test_df["label_id"].tolist(),  tokenizer, MAX_LENGTH)
print(f"Dataset sizes -- Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

Dataset sizes -- Train: 23184, Val: 2898, Test: 2899


Cell 10 — Arsitektur Model (Identik dengan v4) Backbone IndoBERT (768 dimensi) diambil representasi token `[CLS]`-nya (ini representasi ringkasan seluruh kalimat), lalu diteruskan ke *classifier head* custom: Dropout → Linear(256) → GELU → Dropout → Linear(2), menghasilkan skor untuk 2 kelas. Loss function pakai `CrossEntropyLoss` dengan `class_weights` (dari Cell 7) dan `label_smoothing=0.1` — teknik regularisasi supaya model tidak terlalu percaya diri (*overconfident*) terhadap prediksinya sendiri.

In [ ]:
class IndoBERTEnhanced(nn.Module):
    def __init__(self, model_name: str, num_labels: int = 2, dropout: float = 0.2):
        super().__init__()
        self.bert       = AutoModel.from_pretrained(model_name)
        hidden_size     = self.bert.config.hidden_size   # 768

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 256),
            nn.GELU(),
            nn.Dropout(dropout / 2),
            nn.Linear(256, num_labels),
        )

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )
        cls_output = outputs.last_hidden_state[:, 0, :]
        logits     = self.classifier(cls_output)

        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss(
                weight=class_weights_tensor,
                label_smoothing=0.1,
            )
            loss = loss_fct(logits, labels)

        return loss, logits

model = IndoBERTEnhanced(MODEL_NAME, num_labels=2, dropout=0.2)
model.to(DEVICE)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params    : {total_params:,}")
print(f"Trainable params: {trainable_params:,}")

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  498MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Total params    : 124,638,722
Trainable params: 124,638,722


Cell 11 — Hyperparameter Training & DataLoader Menentukan hyperparameter: batch size 16 (dengan gradient accumulation 2x, jadi *effective batch size* 32), maksimum 7 epoch, dan **differential learning rate** — backbone IndoBERT dilatih dengan LR kecil (1e-5) supaya bobot pretrained-nya tidak "rusak", sementara classifier head baru dilatih dengan LR lebih besar (1e-4) karena mulai dari nol. Scheduler mengatur learning rate naik bertahap di awal (*warmup*) lalu turun linear seiring training.

In [ ]:
BATCH_SIZE         = 16
GRAD_ACCUM_STEPS   = 2        # effective batch = 32
EPOCHS             = 7
LR_BERT            = 1e-5
LR_HEAD            = 1e-4
WEIGHT_DECAY       = 0.01
WARMUP_RATIO       = 0.1
GRAD_CLIP          = 1.0
PATIENCE           = 3        # early stopping patience

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=False)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)

# Differential Learning Rate
optimizer = torch.optim.AdamW([
    {"params": model.bert.parameters(),       "lr": LR_BERT, "weight_decay": WEIGHT_DECAY},
    {"params": model.classifier.parameters(), "lr": LR_HEAD, "weight_decay": WEIGHT_DECAY},
])

total_update_steps = (len(train_loader) // GRAD_ACCUM_STEPS) * EPOCHS
warmup_steps       = int(total_update_steps * WARMUP_RATIO)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_update_steps,
)

print(f"Total update steps : {total_update_steps}")
print(f"Warmup steps       : {warmup_steps}")

Total update steps : 5068
Warmup steps       : 506


Cell 12 — Fungsi Evaluasi Fungsi bantu untuk menghitung loss dan metrik (accuracy, precision, recall, F1) pada satu set data. Dipanggil baik untuk data validasi (tiap epoch) maupun data test (di akhir). Model dijalankan dalam mode `eval()` (dropout dimatikan) dan tanpa menghitung gradient (`torch.no_grad()`) — lebih cepat dan hemat memori karena tidak perlu backpropagation.

In [ ]:
def evaluate(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0.0

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["labels"].to(device)

            loss, logits = model(input_ids, attention_mask, labels=labels)
            total_loss  += loss.item()

            preds = torch.argmax(logits, dim=-1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss            = total_loss / len(loader)
    accuracy            = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average="macro", zero_division=0
    )
    return avg_loss, accuracy, precision, recall, f1, all_preds, all_labels

Cell 13 — Training Loop Utama Loop inti training: tiap epoch, model dilatih di seluruh data train, lalu dievaluasi ke data validasi. Kalau F1 validasi membaik dari epoch sebelumnya, bobot model saat itu disimpan sebagai "model terbaik". Kalau F1 tidak membaik selama 3 epoch berturut-turut (`patience=3`), training dihentikan lebih awal (*early stopping*) — mencegah overfitting sekaligus menghemat waktu. Di akhir, bobot **terbaik** (bukan bobot dari epoch terakhir) yang dipakai untuk evaluasi final.

In [ ]:
print("=" * 60)
print("MULAI TRAINING -- IndoBERT v5 (Multi-Kategori, Binary Setup)")
print("=" * 60)

best_val_f1       = 0.0
best_model_state  = None
patience_counter  = 0
history           = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_train_loss = 0.0
    optimizer.zero_grad()

    num_steps = len(train_loader)
    for step, batch in enumerate(train_loader):
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels         = batch["labels"].to(DEVICE)

        loss, _ = model(input_ids, attention_mask, labels=labels)
        loss     = loss / GRAD_ACCUM_STEPS
        loss.backward()
        total_train_loss += loss.item() * GRAD_ACCUM_STEPS

        if (step + 1) % GRAD_ACCUM_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        if step % 20 == 0 or step == num_steps - 1:
            print(f"Epoch {epoch:02d}/{EPOCHS} | Step {step:03d}/{num_steps:03d} | Batch Loss: {loss.item() * GRAD_ACCUM_STEPS:.4f}", flush=True)

    avg_train_loss = total_train_loss / len(train_loader)
    val_loss, val_acc, val_prec, val_rec, val_f1, _, _ = evaluate(model, val_loader, DEVICE)

    history.append({
        "epoch":       epoch,
        "train_loss":  avg_train_loss,
        "val_loss":    val_loss,
        "val_acc":     val_acc,
        "val_f1":      val_f1,
    })

    flag = ""
    if val_f1 > best_val_f1:
        best_val_f1      = val_f1
        best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
        patience_counter = 0
        flag = " << BEST"
    else:
        patience_counter += 1
        flag = f" -- patience {patience_counter}/{PATIENCE}"

    print(
        f"Epoch {epoch:02d}/{EPOCHS}  "
        f"| Train Loss: {avg_train_loss:.4f} "
        f"| Val Loss: {val_loss:.4f} "
        f"| Val Acc: {val_acc:.4f} "
        f"| Val F1: {val_f1:.4f}"
        f"{flag}",
        flush=True,
    )

    if patience_counter >= PATIENCE:
        print("\nEarly stopping -- val F1 tidak membaik selama 3 epoch.")
        break

print(f"\nTraining selesai. Best Val F1: {best_val_f1:.4f}")
model.load_state_dict(best_model_state)
print("Best model weights di-load kembali.")

MULAI TRAINING -- IndoBERT v5 (Multi-Kategori, Binary Setup)
Epoch 01/7 | Step 000/1449 | Batch Loss: 0.6876
Epoch 01/7 | Step 020/1449 | Batch Loss: 0.6743
Epoch 01/7 | Step 040/1449 | Batch Loss: 0.5998
Epoch 01/7 | Step 060/1449 | Batch Loss: 0.6504
Epoch 01/7 | Step 080/1449 | Batch Loss: 0.5921
Epoch 01/7 | Step 100/1449 | Batch Loss: 0.5545
Epoch 01/7 | Step 120/1449 | Batch Loss: 0.5614
Epoch 01/7 | Step 140/1449 | Batch Loss: 0.6099
Epoch 01/7 | Step 160/1449 | Batch Loss: 0.3519
Epoch 01/7 | Step 180/1449 | Batch Loss: 0.3606
Epoch 01/7 | Step 200/1449 | Batch Loss: 0.4795
Epoch 01/7 | Step 220/1449 | Batch Loss: 0.2615
Epoch 01/7 | Step 240/1449 | Batch Loss: 0.5471
Epoch 01/7 | Step 260/1449 | Batch Loss: 0.4508
Epoch 01/7 | Step 280/1449 | Batch Loss: 0.2291
Epoch 01/7 | Step 300/1449 | Batch Loss: 0.2505
Epoch 01/7 | Step 320/1449 | Batch Loss: 0.2176
Epoch 01/7 | Step 340/1449 | Batch Loss: 0.3210
Epoch 01/7 | Step 360/1449 | Batch Loss: 0.3560
Epoch 01/7 | Step 380/1449 

Cell 14 — Evaluasi Final di Test Set Model (dengan bobot terbaik) dievaluasi sekali terhadap data test yang sama sekali belum pernah dilihat selama training maupun validasi — inilah angka yang jadi hasil akhir untuk dilaporkan di Bab 4 skripsi. Ditampilkan *classification report* (precision/recall/F1 per kelas) dan *confusion matrix* untuk melihat detail jenis kesalahan model.

In [ ]:
print("=" * 60)
print("TEST SET EVALUATION -- v5 (Multi-Kategori, Binary)")
print("=" * 60)

test_loss, test_acc, test_prec, test_rec, test_f1, y_pred, y_true = evaluate(
    model, test_loader, DEVICE
)

print(f"Test Accuracy        : {test_acc:.4f}")
print(f"Test Precision (mac) : {test_prec:.4f}")
print(f"Test Recall    (mac) : {test_rec:.4f}")
print(f"Test F1        (mac) : {test_f1:.4f}")

print("\n-- Classification Report ----------------------------------------")
print(classification_report(y_true, y_pred, target_names=[ID2LABEL[i] for i in range(2)], zero_division=0))

print("-- Confusion Matrix ---------------------------------------------")
cm = confusion_matrix(y_true, y_pred)
print(cm)

print("\n-- Training History ---------------------------------------------")
history_df = pd.DataFrame(history)
print(history_df.to_string(index=False))

TEST SET EVALUATION -- v5 (Multi-Kategori, Binary)
Test Accuracy        : 0.9779
Test Precision (mac) : 0.9781
Test Recall    (mac) : 0.9774
Test F1        (mac) : 0.9777

-- Classification Report ----------------------------------------
              precision    recall  f1-score   support

    negative       0.98      0.98      0.98      1574
    positive       0.98      0.97      0.98      1325

    accuracy                           0.98      2899
   macro avg       0.98      0.98      0.98      2899
weighted avg       0.98      0.98      0.98      2899

-- Confusion Matrix ---------------------------------------------
[[1548   26]
 [  38 1287]]

-- Training History ---------------------------------------------
 epoch  train_loss  val_loss  val_acc   val_f1
     1    0.303471  0.259456 0.966529 0.966370
     2    0.236141  0.255945 0.968599 0.968459
     3    0.217159  0.264442 0.968599 0.968462
     4    0.209140  0.250577 0.976190 0.976046
     5    0.205971  0.249094 0.978261 0.

Cell 15 — Evaluasi Per Kategori Aplikasi Karena dataset mencakup 17 kategori aplikasi berbeda, di sini performa model dihitung **terpisah per kategori** (bukan cuma rata-rata keseluruhan) — supaya kelihatan apakah model konsisten bagus di semua domain aplikasi, atau ada kategori tertentu (misal game, yang bahasanya lebih santai dan penuh istilah khusus) yang jauh lebih sulit dikenali dibanding kategori lain.

In [ ]:
eval_df = test_df.copy()
eval_df["y_true"] = y_true
eval_df["y_pred"] = y_pred
eval_df["benar"]  = eval_df["y_true"] == eval_df["y_pred"]

print("=" * 70)
print("PERFORMA MODEL PER KATEGORI APLIKASI")
print("=" * 70)

rows = []
for kategori, grp in eval_df.groupby(KATEGORI_COL):
    acc = accuracy_score(grp["y_true"], grp["y_pred"])
    prec, rec, f1, _ = precision_recall_fscore_support(
        grp["y_true"], grp["y_pred"], average="macro", zero_division=0
    )
    rows.append({
        "kategori": kategori,
        "n_data": len(grp),
        "accuracy": round(acc, 4),
        "precision_macro": round(prec, 4),
        "recall_macro": round(rec, 4),
        "f1_macro": round(f1, 4),
    })

per_kategori_df = pd.DataFrame(rows).sort_values("f1_macro", ascending=False).reset_index(drop=True)
print(per_kategori_df.to_string(index=False))

print(f"\nKategori dengan performa TERBAIK : {per_kategori_df.iloc[0]['kategori']} (F1={per_kategori_df.iloc[0]['f1_macro']})")
print(f"Kategori dengan performa TERLEMAH : {per_kategori_df.iloc[-1]['kategori']} (F1={per_kategori_df.iloc[-1]['f1_macro']})")

per_kategori_df.to_csv("evaluasi_per_kategori_v5.csv", index=False)
print("\nDisimpan ke: evaluasi_per_kategori_v5.csv")

PERFORMA MODEL PER KATEGORI APLIKASI
             kategori  n_data  accuracy  precision_macro  recall_macro  f1_macro
      kesehatan_medis     167    1.0000           1.0000        1.0000    1.0000
            ecommerce     215    0.9953           0.9959        0.9947    0.9953
   keuangan_investasi     199    0.9899           0.9897        0.9897    0.9897
  transportasi_travel     178    0.9888           0.9888        0.9890    0.9888
            perbankan     177    0.9887           0.9885        0.9885    0.9885
       dompet_digital     183    0.9891           0.9857        0.9857    0.9857
    hiburan_streaming     201    0.9851           0.9857        0.9848    0.9851
           komunikasi     212    0.9906           0.9944        0.9730    0.9833
           pendidikan     122    0.9836           0.9822        0.9822    0.9822
   game_casual_puzzle     162    0.9691           0.9675        0.9722    0.9689
gaya_hidup_kecantikan     180    0.9667           0.9684        0.9656  

Cell 16 — Simpan Model Menyimpan semua yang dibutuhkan supaya model bisa dipakai lagi tanpa training ulang: bobot model (`model_weights.pt`), file tokenizer, konfigurasi arsitektur, riwayat training per epoch, hasil evaluasi per kategori, dan metadata ringkasan (versi model, dataset yang dipakai, metrik akhir). File-file inilah yang nantinya dipindahkan ke folder chatbot untuk menggantikan model v4 lama di `predictor.py`.

In [ ]:
OUTPUT_DIR = "indobert_multikategori_v5"
os.makedirs(OUTPUT_DIR, exist_ok=True)

torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, "model_weights.pt"))
tokenizer.save_pretrained(OUTPUT_DIR)
model.bert.config.save_pretrained(OUTPUT_DIR)

history_df.to_csv(os.path.join(OUTPUT_DIR, "training_history.csv"), index=False)
per_kategori_df.to_csv(os.path.join(OUTPUT_DIR, "evaluasi_per_kategori.csv"), index=False)

metadata = {
    "model_version": "v5",
    "dataset": "ulasan_aplikasi_50000_relabeled.csv",
    "cakupan": "17 kategori aplikasi (bukan hanya e-commerce)",
    "labeling_method": "text-based + emoji mapping (hybrid correction dari rating)",
    "class_imbalance_handling": "class weights (balanced) -- BUKAN SMOTE",
    "test_accuracy": test_acc,
    "test_f1_macro": test_f1,
    "test_precision_macro": test_prec,
    "test_recall_macro": test_rec,
    "epochs_trained": len(history),
    "best_val_f1": best_val_f1,
}
pd.DataFrame([metadata]).to_csv(os.path.join(OUTPUT_DIR, "model_metadata.csv"), index=False)

print(f"Model v5 tersimpan di: {OUTPUT_DIR}")
print(f"  - model_weights.pt")
print(f"  - tokenizer files")
print(f"  - training_history.csv")
print(f"  - evaluasi_per_kategori.csv")
print(f"  - model_metadata.csv")

Model v5 tersimpan di: indobert_multikategori_v5
  - model_weights.pt
  - tokenizer files
  - training_history.csv
  - evaluasi_per_kategori.csv
  - model_metadata.csv


Cell 17 — Fungsi Inference (Prediksi Teks Baru) Fungsi siap pakai untuk memprediksi sentimen dari teks ulasan baru (di luar dataset training/test) — dipakai untuk demo di notebook ini, dan jadi acuan logika yang diadaptasi ke `predictor.py` di sisi backend chatbot.

In [ ]:
def predict_sentiment(texts: list, model, tokenizer, device, max_length=256):
    """
    Prediksi sentimen untuk ulasan teks kustom (lintas kategori aplikasi apapun).
    """
    model.eval()
    results = []

    batch_size = 32
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]
        clean_batch = [clean_text(t) for t in batch_texts]
        enc = tokenizer(
            clean_batch,
            truncation=True,
            padding=True,
            max_length=max_length,
            return_tensors="pt",
        )
        input_ids      = enc["input_ids"].to(device)
        attention_mask = enc["attention_mask"].to(device)

        with torch.no_grad():
            _, logits = model(input_ids, attention_mask)
            probs = torch.softmax(logits, dim=-1)

        for j, text in enumerate(batch_texts):
            scores     = probs[j].cpu().numpy()
            label_id   = int(scores.argmax())
            confidence = float(scores.max())
            results.append({
                "text":       text,
                "label":      ID2LABEL[label_id],
                "label_id":   label_id,
                "confidence": confidence,
                "scores": {
                    "negative": float(scores[0]),
                    "positive": float(scores[1]),
                },
            })
    return results

Cell 18 — Demo Inference Lintas Kategori Contoh pemakaian `predict_sentiment()` dengan beberapa ulasan dari kategori berbeda (e-commerce, perbankan, kesehatan, game, transportasi) — sebagai bukti kualitatif bahwa model bisa menangani berbagai domain aplikasi, bukan cuma satu kategori tertentu.

In [ ]:
sample_texts = [
    "shopee bagus banget 👍, pengirimannya cepat dan barangnya sampai dalam kondisi sempurna!",
    "aplikasi bca mobile sering error pas mau transfer, saldo kepotong tapi transaksi gagal",
    "halodoc sangat membantu, konsultasi dokter cepat dan responnya jelas banget",
    "mobile legends makin toxic, banyak hacker dan server sering down",
    "gojek driver nya ramah banget, tepat waktu dan harganya terjangkau",
]

print("-- Contoh Inference v5 (Lintas Kategori) ---------")
preds = predict_sentiment(sample_texts, model, tokenizer, DEVICE)
for p in preds:
    print(f"\nText       : {p['text']}")
    print(f"   Prediksi   : {p['label'].upper()} (confidence: {p['confidence']:.3f})")
    print(f"   Scores     : {p['scores']}")

-- Contoh Inference v5 (Lintas Kategori) ---------

Text       : shopee bagus banget 👍, pengirimannya cepat dan barangnya sampai dalam kondisi sempurna!
   Prediksi   : POSITIVE (confidence: 0.958)
   Scores     : {'negative': 0.0417436845600605, 'positive': 0.9582562446594238}

Text       : aplikasi bca mobile sering error pas mau transfer, saldo kepotong tapi transaksi gagal
   Prediksi   : NEGATIVE (confidence: 0.940)
   Scores     : {'negative': 0.9397276043891907, 'positive': 0.06027241051197052}

Text       : halodoc sangat membantu, konsultasi dokter cepat dan responnya jelas banget
   Prediksi   : POSITIVE (confidence: 0.959)
   Scores     : {'negative': 0.04082488268613815, 'positive': 0.9591751098632812}

Text       : mobile legends makin toxic, banyak hacker dan server sering down
   Prediksi   : NEGATIVE (confidence: 0.942)
   Scores     : {'negative': 0.9415520429611206, 'positive': 0.05844803526997566}

Text       : gojek driver nya ramah banget, tepat waktu dan harganya 

Cell Tambahan A — Diagnostik: Akurasi per Rentang Panjang Ulasan Cell ini ditambahkan di luar notebook awal, untuk menyelidiki kenapa akurasi test set (97,79%) terasa mencurigakan tinggi. Data test dipecah ke beberapa kelompok berdasarkan jumlah kata (1 kata, 2-3 kata, 4-7 kata, dst), lalu dihitung akurasi model di tiap kelompok — untuk mengecek apakah ulasan pendek atau generik (mis. "bagus", "mantap") yang membuat angka akurasi keseluruhan kelihatan tinggi, atau performa memang konsisten di semua panjang teks.

In [ ]:
eval_df = test_df.copy()
eval_df["y_true"] = y_true
eval_df["y_pred"] = y_pred
eval_df["benar"] = eval_df["y_true"] == eval_df["y_pred"]
eval_df["n_kata"] = eval_df[TEXT_COL].str.split().apply(len)

bins = [(0,1,"1 kata"), (2,3,"2-3 kata"), (4,7,"4-7 kata"), (8,15,"8-15 kata"), (16,999,">15 kata")]
print(f"{'Segmen':<12} {'n_data':>8} {'Akurasi':>10}")
for lo, hi, label in bins:
    subset = eval_df[(eval_df["n_kata"]>=lo) & (eval_df["n_kata"]<=hi)]
    if len(subset) > 0:
        print(f"{label:<12} {len(subset):>8} {subset['benar'].mean():>10.4f}")

Segmen         n_data    Akurasi
1 kata             76     0.9474
2-3 kata          482     0.9751
4-7 kata          739     0.9824
8-15 kata         713     0.9846
>15 kata          889     0.9730


Cell Tambahan B — Prediksi Model v5 pada Sample Validasi Manual Cell ini menjalankan model v5 (dengan bobot yang sudah dilatih di notebook ini) terhadap 250 sample yang sama persis dengan yang sebelumnya dilabel manual oleh manusia. Hasilnya disimpan sebagai kolom `label_model_v5`, supaya bisa dibandingkan langsung: apakah prediksi model v5 lebih dekat ke penilaian manusia, atau cuma meniru label dari model pretrained (`mdhugol`) yang dipakai untuk labeling — ini jadi bukti kunci untuk menguji dugaan *self-distillation*.

In [ ]:
# Load 250 teks yang sama persis dari validasi manual
validasi_texts_df = pd.read_csv("validasi_manual_BLIND.csv")

preds = predict_sentiment(validasi_texts_df["ulasan"].tolist(), model, tokenizer, DEVICE)
validasi_texts_df["label_model_v5"] = [p["label"].replace("positive", "positif").replace("negative", "negatif") for p in preds]

validasi_texts_df.to_csv("validasi_manual_dengan_prediksi_v5.csv", index=False)
print("Selesai, upload file ini ke Claude untuk dihitung.")

Selesai, upload file ini ke Claude untuk dihitung.
